In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 80 — Benchmark Lab: Same Use Case Across Frameworks

## Goal
Implement the **same RAG Q&A task** in **LlamaIndex**,
**Haystack**, and **DSPy**, then compare **code complexity**,
**answer quality**, and **latency**.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Framework comparison** | Same task in 3 styles |
| **RAG patterns** | Each framework's RAG approach |
| **Evaluation** | Consistent metrics across frameworks |
| **Trade-offs** | Verbosity, flexibility, optimization |

## Shared Setup
- Same knowledge base (5 science docs)
- Same LLM: Ollama `qwen3.5:9b`
- Same embeddings: `BAAI/bge-small-en-v1.5`
- Same 5 test questions with reference answers

In [ ]:
import os, warnings, time
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

print("Environment configured")

## Shared Knowledge Base & Test Queries

In [ ]:
# Shared knowledge base — 5 science documents
KNOWLEDGE_BASE = [
    {
        "id": "photosynthesis",
        "text": """Photosynthesis is the process by which plants convert sunlight,
water, and carbon dioxide into glucose and oxygen. It occurs primarily in
chloroplasts using chlorophyll pigment. The light-dependent reactions happen
in the thylakoid membranes, producing ATP and NADPH. The Calvin cycle
(light-independent reactions) uses these to fix carbon dioxide into glucose.
Photosynthesis is responsible for producing most of Earth's oxygen."""
    },
    {
        "id": "gravity",
        "text": """Gravity is a fundamental force of nature that attracts objects
with mass toward each other. Newton described it as F = G*m1*m2/r^2.
Einstein's General Relativity explains gravity as the curvature of spacetime
caused by mass and energy. On Earth, gravitational acceleration is approximately
9.8 m/s^2. Gravity keeps planets in orbit, causes tides, and is the weakest
of the four fundamental forces."""
    },
    {
        "id": "dna",
        "text": """DNA (deoxyribonucleic acid) is a double-helix molecule that
carries genetic instructions for all living organisms. It consists of four
nucleotide bases: adenine (A), thymine (T), guanine (G), and cytosine (C).
A pairs with T, and G pairs with C (base pairing). DNA replication is
semi-conservative: each strand serves as a template. The human genome contains
approximately 3 billion base pairs and about 20,000-25,000 genes."""
    },
    {
        "id": "climate",
        "text": """Climate change refers to long-term shifts in global temperatures
and weather patterns. Since the Industrial Revolution, human activities
(burning fossil fuels, deforestation) have increased atmospheric CO2 from
280 ppm to over 420 ppm. The greenhouse effect traps heat in the atmosphere.
Effects include rising sea levels, more extreme weather events, and changes
in ecosystems. The Paris Agreement aims to limit warming to 1.5°C above
pre-industrial levels."""
    },
    {
        "id": "neurons",
        "text": """Neurons are the fundamental units of the nervous system.
They transmit information through electrical and chemical signals.
A neuron consists of a cell body (soma), dendrites (receive signals),
and an axon (transmits signals). Signal transmission across synapses
involves neurotransmitters like dopamine, serotonin, and acetylcholine.
The human brain contains approximately 86 billion neurons forming
trillions of synaptic connections."""
    },
]

# Test queries with reference answers (for evaluation)
TEST_QUERIES = [
    {
        "question": "Where does photosynthesis occur in the cell?",
        "expected_keywords": ["chloroplast", "thylakoid"],
        "relevant_doc": "photosynthesis"
    },
    {
        "question": "What is the formula for gravitational force?",
        "expected_keywords": ["F", "G", "m1", "m2", "r"],
        "relevant_doc": "gravity"
    },
    {
        "question": "How many base pairs are in the human genome?",
        "expected_keywords": ["3", "billion"],
        "relevant_doc": "dna"
    },
    {
        "question": "What is the current atmospheric CO2 level?",
        "expected_keywords": ["420", "ppm"],
        "relevant_doc": "climate"
    },
    {
        "question": "How many neurons does the human brain have?",
        "expected_keywords": ["86", "billion"],
        "relevant_doc": "neurons"
    },
]

def keyword_score(answer: str, expected: list) -> float:
    """Fraction of expected keywords found in the answer."""
    answer_lower = answer.lower()
    found = sum(1 for kw in expected if kw.lower() in answer_lower)
    return found / len(expected) if expected else 0.0

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents")
print(f"Test queries: {len(TEST_QUERIES)} questions")

---
## Framework 1: LlamaIndex

LlamaIndex provides a **high-level** API:
create documents → build index → query.
Minimal code, batteries included.

In [ ]:
from llama_index.core import VectorStoreIndex, Document as LIDoc, Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Configure once
Settings.llm = Ollama(model="qwen3.5:9b", request_timeout=120.0, temperature=0)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.chunk_size = 512

# Build index
li_docs = [LIDoc(text=d["text"], metadata={"id": d["id"]}) for d in KNOWLEDGE_BASE]
li_index = VectorStoreIndex.from_documents(li_docs)
li_engine = li_index.as_query_engine(similarity_top_k=2)

def llamaindex_rag(question: str) -> str:
    response = li_engine.query(question)
    return str(response)

print("LlamaIndex RAG ready")

In [ ]:
# Run LlamaIndex benchmark
print("\n" + "="*60)
print("LLAMAINDEX BENCHMARK")
print("="*60)

li_results = []
li_total_time = 0

for tq in TEST_QUERIES:
    t0 = time.time()
    answer = llamaindex_rag(tq["question"])
    elapsed = time.time() - t0
    li_total_time += elapsed
    
    score = keyword_score(answer, tq["expected_keywords"])
    li_results.append({"question": tq["question"], "score": score, "time": elapsed})
    
    print(f"\nQ: {tq['question']}")
    print(f"A: {answer[:200]}")
    print(f"Score: {score:.0%} | Time: {elapsed:.2f}s")

li_avg_score = sum(r["score"] for r in li_results) / len(li_results)
print(f"\n--- LlamaIndex Summary: Avg Score={li_avg_score:.0%} | Total Time={li_total_time:.1f}s ---")

---
## Framework 2: Haystack

Haystack uses an **explicit pipeline** approach:
you wire components together and run the pipeline.
More control over each step.

In [ ]:
from haystack import Document as HDoc
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder
)
from haystack.components.builders import PromptBuilder
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
from haystack.dataclasses import ChatMessage

# Document store + embeddings
hs_store = InMemoryDocumentStore()
hs_docs = [HDoc(content=d["text"], meta={"id": d["id"]}) for d in KNOWLEDGE_BASE]

embedder = SentenceTransformersDocumentEmbedder(model="BAAI/bge-small-en-v1.5")
embedder.warm_up()
embedded = embedder.run(documents=hs_docs)
hs_store.write_documents(embedded["documents"])

# Components
text_embedder = SentenceTransformersTextEmbedder(model="BAAI/bge-small-en-v1.5")
text_embedder.warm_up()
hs_retriever = InMemoryEmbeddingRetriever(document_store=hs_store, top_k=2)

HS_TEMPLATE = """Answer the question based on the context below.

Context:
{% for doc in documents %}
{{ doc.content }}
{% endfor %}

Question: {{ question }}
Answer:"""

hs_prompt_builder = PromptBuilder(template=HS_TEMPLATE)
hs_generator = OllamaChatGenerator(
    model="qwen3.5:9b",
    url="http://localhost:11434",
    generation_kwargs={"temperature": 0, "num_predict": 256}
)

def haystack_rag(question: str) -> str:
    # Step-by-step execution
    query_emb = text_embedder.run(text=question)
    docs = hs_retriever.run(query_embedding=query_emb["embedding"])
    prompt = hs_prompt_builder.run(documents=docs["documents"], question=question)
    messages = [ChatMessage.from_user(prompt["prompt"])]
    result = hs_generator.run(messages=messages)
    reply = result["replies"][0].text
    # Strip thinking tags
    if "<think>" in reply:
        reply = reply.split("</think>")[-1].strip()
    return reply

print("Haystack RAG ready")

In [ ]:
# Run Haystack benchmark
print("\n" + "="*60)
print("HAYSTACK BENCHMARK")
print("="*60)

hs_results = []
hs_total_time = 0

for tq in TEST_QUERIES:
    t0 = time.time()
    answer = haystack_rag(tq["question"])
    elapsed = time.time() - t0
    hs_total_time += elapsed
    
    score = keyword_score(answer, tq["expected_keywords"])
    hs_results.append({"question": tq["question"], "score": score, "time": elapsed})
    
    print(f"\nQ: {tq['question']}")
    print(f"A: {answer[:200]}")
    print(f"Score: {score:.0%} | Time: {elapsed:.2f}s")

hs_avg_score = sum(r["score"] for r in hs_results) / len(hs_results)
print(f"\n--- Haystack Summary: Avg Score={hs_avg_score:.0%} | Total Time={hs_total_time:.1f}s ---")

---
## Framework 3: DSPy

DSPy treats the pipeline as a **program** with
optimizable prompts. You define signatures and modules;
DSPy manages the prompt engineering.

In [ ]:
import dspy

lm = dspy.LM(
    "ollama_chat/qwen3.5:9b",
    api_base="http://localhost:11434",
    api_key="fake",
    temperature=0,
    max_tokens=256
)
dspy.configure(lm=lm)

# Simple keyword retriever (same ranking for fair comparison)
from sentence_transformers import SentenceTransformer
import numpy as np

st_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
kb_texts = [d["text"] for d in KNOWLEDGE_BASE]
kb_embeddings = st_model.encode(kb_texts, normalize_embeddings=True)

def dspy_retrieve(query: str, top_k: int = 2) -> list:
    q_emb = st_model.encode([query], normalize_embeddings=True)
    scores = np.dot(kb_embeddings, q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [kb_texts[i] for i in top_idx]

# DSPy RAG module
class DSPyRAG(dspy.Module):
    def __init__(self):
        self.respond = dspy.ChainOfThought("context, question -> answer")
    
    def forward(self, question: str) -> dspy.Prediction:
        docs = dspy_retrieve(question)
        context = "\n\n".join(docs)
        return self.respond(context=context, question=question)

dspy_rag = DSPyRAG()

def dspy_rag_fn(question: str) -> str:
    result = dspy_rag(question=question)
    answer = result.answer
    # Strip thinking tags
    if "<think>" in answer:
        answer = answer.split("</think>")[-1].strip()
    return answer

print("DSPy RAG ready")

In [ ]:
# Run DSPy benchmark
print("\n" + "="*60)
print("DSPY BENCHMARK")
print("="*60)

dspy_results = []
dspy_total_time = 0

for tq in TEST_QUERIES:
    t0 = time.time()
    answer = dspy_rag_fn(tq["question"])
    elapsed = time.time() - t0
    dspy_total_time += elapsed
    
    score = keyword_score(answer, tq["expected_keywords"])
    dspy_results.append({"question": tq["question"], "score": score, "time": elapsed})
    
    print(f"\nQ: {tq['question']}")
    print(f"A: {answer[:200]}")
    print(f"Score: {score:.0%} | Time: {elapsed:.2f}s")

dspy_avg_score = sum(r["score"] for r in dspy_results) / len(dspy_results)
print(f"\n--- DSPy Summary: Avg Score={dspy_avg_score:.0%} | Total Time={dspy_total_time:.1f}s ---")

---
## Side-by-Side Comparison

In [ ]:
print("\n" + "="*70)
print("CROSS-FRAMEWORK COMPARISON")
print("="*70)

frameworks = {
    "LlamaIndex": {"results": li_results, "total_time": li_total_time, "avg_score": li_avg_score},
    "Haystack":   {"results": hs_results, "total_time": hs_total_time, "avg_score": hs_avg_score},
    "DSPy":       {"results": dspy_results, "total_time": dspy_total_time, "avg_score": dspy_avg_score},
}

# Per-question comparison
print(f"\n{'Question':<50} {'LlamaIndex':>10} {'Haystack':>10} {'DSPy':>10}")
print("-" * 82)
for i, tq in enumerate(TEST_QUERIES):
    q = tq['question'][:47] + '...' if len(tq['question']) > 50 else tq['question']
    li_s = f"{li_results[i]['score']:.0%}"
    hs_s = f"{hs_results[i]['score']:.0%}"
    ds_s = f"{dspy_results[i]['score']:.0%}"
    print(f"{q:<50} {li_s:>10} {hs_s:>10} {ds_s:>10}")

# Summary table
print(f"\n{'Metric':<25} {'LlamaIndex':>12} {'Haystack':>12} {'DSPy':>12}")
print("-" * 63)
print(f"{'Avg Keyword Score':<25} {li_avg_score:>11.0%} {hs_avg_score:>11.0%} {dspy_avg_score:>11.0%}")
print(f"{'Total Time (s)':<25} {li_total_time:>11.1f} {hs_total_time:>11.1f} {dspy_total_time:>11.1f}")
avg_li = li_total_time / len(li_results)
avg_hs = hs_total_time / len(hs_results)
avg_ds = dspy_total_time / len(dspy_results)
print(f"{'Avg Time/Query (s)':<25} {avg_li:>11.2f} {avg_hs:>11.2f} {avg_ds:>11.2f}")

In [ ]:
# Code complexity comparison
print("\n" + "="*70)
print("CODE COMPLEXITY COMPARISON")
print("="*70)

comparison = [
    ["Setup lines",       "~6",  "~15", "~12"],
    ["Query function",    "~3",  "~8",  "~5"],
    ["Prompt control",    "Low (auto)",  "High (Jinja2)", "Medium (signatures)"],
    ["Retriever",         "Built-in",  "Explicit wiring", "Custom function"],
    ["Optimization",      "Manual",  "Manual", "Built-in (LabeledFewShot, MIPRO)"],
    ["Pipeline visibility","Abstracted","Fully visible", "Module graph"],
    ["Learning curve",    "Low",  "Medium", "Medium-High"],
    ["Best for",          "Quick prototyping", "Production pipelines", "Prompt optimization"],
]

print(f"\n{'Aspect':<22} {'LlamaIndex':<18} {'Haystack':<20} {'DSPy':<22}")
print("-" * 82)
for row in comparison:
    print(f"{row[0]:<22} {row[1]:<18} {row[2]:<20} {row[3]:<22}")

## 🧠 Key Takeaways

### When to Use Each Framework

| Framework | Best For |
|-----------|----------|
| **LlamaIndex** | Fastest prototyping, data connectors, simple RAG |
| **Haystack** | Production pipelines, component swapping, enterprise |
| **DSPy** | Prompt optimization, research, when you want auto-tuning |

### Architecture Comparison
```
LlamaIndex:  Documents → Index → QueryEngine → Answer
             (high-level, fewest lines)

Haystack:    Store → Retriever → PromptBuilder → Generator
             (explicit pipeline, most control)

DSPy:        Module(Signature) → forward() → Prediction
             (programmatic, optimizable prompts)
```

### Practical Advice
- **Start with LlamaIndex** for prototyping and exploration
- **Move to Haystack** when you need production-grade pipelines
- **Use DSPy** when prompt quality is the bottleneck
- All three can use the same underlying LLM and embeddings
- They are **not mutually exclusive** — you can use DSPy modules
  inside a Haystack pipeline, or LlamaIndex retrievers with DSPy